# 08 - ML Life Expectancy Prediction and Country Overperformance Analysis

This notebook uses regularized ML models to predict life expectancy from 10 country-level
features, then examines **residuals** (actual - predicted) to identify countries that
significantly outperform or underperform their predicted life expectancy.

Countries with large positive residuals are "hidden Blue Zone" candidates -- places where
unmeasured factors (culture, diet, social cohesion) may boost longevity beyond what
standard indicators predict.

**Important caveat:** With n=93 countries, we use LOOCV (Leave-One-Out Cross-Validation)
for honest out-of-sample evaluation and regularized models to prevent overfitting.
This is correlation-based pattern detection, not causal inference.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

PROJECT_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
    PROJECT_DIR = os.getcwd()
    if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
        PROJECT_DIR = os.path.dirname(PROJECT_DIR)

ANALYSIS_DIR = os.path.join(PROJECT_DIR, 'outputs', 'analysis')
FIGURES_DIR = os.path.join(PROJECT_DIR, 'outputs', 'figures')

# Load all ML outputs
model_comp = pd.read_csv(os.path.join(ANALYSIS_DIR, 'ml_model_comparison.csv'))
feat_imp = pd.read_csv(os.path.join(ANALYSIS_DIR, 'ml_feature_importance.csv'))
feat_sel = pd.read_csv(os.path.join(ANALYSIS_DIR, 'ml_feature_selection_report.csv'))
residuals = pd.read_csv(os.path.join(ANALYSIS_DIR, 'ml_residual_analysis.csv'))
hidden_bz = pd.read_csv(os.path.join(ANALYSIS_DIR, 'ml_hidden_blue_zones.csv'))
underperf = pd.read_csv(os.path.join(ANALYSIS_DIR, 'ml_underperformers.csv'))
features = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'features', 'ml_feature_matrix.csv'))

BLUE_ZONE_ISOS = {'USA', 'JPN', 'ITA', 'GRC', 'CRI'}
print('Data loaded successfully')
print(f'Feature matrix: {features.shape}')
print(f'Models compared: {len(model_comp)}')
print(f'Countries analyzed: {len(residuals)}')

Data loaded successfully
Feature matrix: (93, 36)
Models compared: 5
Countries analyzed: 93


## 1. Feature Selection Report

Starting from 26 candidate features, a three-stage pipeline selects the final feature set:
1. **Coverage filter:** Drop features with <50% non-null values across 93 countries
2. **Correlation filter:** Among highly correlated pairs (|r| > 0.85), drop the weaker predictor
3. **VIF filter:** Iteratively drop features with variance inflation factor > 10

In [2]:
# Feature selection summary
kept = feat_sel[feat_sel['stage_dropped'] == 'kept']
dropped = feat_sel[feat_sel['stage_dropped'] != 'kept']

print(f'Features entering pipeline: {len(feat_sel)}')
print(f'Features surviving all stages: {len(kept)}')
print(f'Features dropped: {len(dropped)}')
print()

# Show what was dropped and why
for stage in ['coverage', 'correlation', 'vif']:
    stage_dropped = dropped[dropped['stage_dropped'] == stage]
    if len(stage_dropped) > 0:
        print(f'\nDropped at {stage.upper()} stage ({len(stage_dropped)}):')
        for _, row in stage_dropped.iterrows():
            print(f'  {row["feature"]:30s} {row["reason"]}')

print(f'\nFinal features ({len(kept)}):')
for _, row in kept.iterrows():
    print(f'  {row["feature"]:30s} (coverage: {row["coverage_pct"]:.0f}%)')

Features entering pipeline: 41
Features surviving all stages: 25
Features dropped: 16


Dropped at COVERAGE stage (1):
  internet_users_pct             Only 38.7% non-null (threshold: 50%)

Dropped at CORRELATION stage (5):
  clean_water_access_pct         |r|=0.867 with log_gdp_per_capita; lower target corr
  health_expenditure_pc          |r|=0.921 with gdp_per_capita; lower target corr
  gdp_per_capita                 |r|=0.979 with gni_per_capita; lower target corr
  gni_per_capita                 |r|=0.853 with log_gdp_per_capita; lower target corr
  mean_temperature               |r|=0.914 with abs_latitude; lower target corr

Dropped at VIF stage (10):
  log_gdp_per_capita             VIF=1203.7 (threshold: 10)
  gini_index                     VIF=902.8 (threshold: 10)
  adult_literacy_rate            VIF=251.0 (threshold: 10)
  population_65plus_pct          VIF=70.5 (threshold: 10)
  measles_immunization           VIF=46.0 (threshold: 10)
  death_rate                     VIF=3

## 2. Model Comparison

In [3]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = model_comp['model'].values
colors = ['#2196F3' if r2 == model_comp['loocv_r2'].max() else '#90CAF9'
          for r2 in model_comp['loocv_r2']]

# R2
axes[0].barh(models, model_comp['loocv_r2'], color=colors)
axes[0].set_xlabel('LOOCV R-squared')
axes[0].set_title('Model R-squared (higher = better)')
axes[0].axvline(x=0.706, color='red', linestyle='--', alpha=0.7, label='Baseline OLS (R2=0.706)')
axes[0].legend(fontsize=8)
for i, v in enumerate(model_comp['loocv_r2']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)

# RMSE
colors_rmse = ['#FF9800' if rmse == model_comp['loocv_rmse'].min() else '#FFE0B2'
               for rmse in model_comp['loocv_rmse']]
axes[1].barh(models, model_comp['loocv_rmse'], color=colors_rmse)
axes[1].set_xlabel('LOOCV RMSE (years)')
axes[1].set_title('Model RMSE (lower = better)')
for i, v in enumerate(model_comp['loocv_rmse']):
    axes[1].text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)

# MAE
colors_mae = ['#4CAF50' if mae == model_comp['loocv_mae'].min() else '#C8E6C9'
              for mae in model_comp['loocv_mae']]
axes[2].barh(models, model_comp['loocv_mae'], color=colors_mae)
axes[2].set_xlabel('LOOCV MAE (years)')
axes[2].set_title('Model MAE (lower = better)')
for i, v in enumerate(model_comp['loocv_mae']):
    axes[2].text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=9)

plt.suptitle('ML Model Comparison (Leave-One-Out Cross-Validation, n=93)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'nb08_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nModel Comparison Table:')
print(model_comp.to_string(index=False))
print(f'\nBest model: {model_comp.iloc[0]["model"]} (LOOCV R2 = {model_comp.iloc[0]["loocv_r2"]:.4f})')


Model Comparison Table:


        model  loocv_r2  loocv_rmse  loocv_mae  n_features
Random Forest    0.6832       3.562      2.714          10
        Ridge    0.6793       3.584      2.716          10
   ElasticNet    0.6783       3.589      2.715          10
 OLS Extended    0.6746       3.610      2.769          10
        Lasso    0.6363       3.817      2.915           6

Best model: Random Forest (LOOCV R2 = 0.6832)


/tmp/ipykernel_1629084/1896755410.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Feature Importance

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Lasso coefficients
feat_sorted_lasso = feat_imp.sort_values('lasso_coef')
colors_lasso = ['#E74C3C' if c < 0 else '#2ECC71' for c in feat_sorted_lasso['lasso_coef']]
ax1.barh(feat_sorted_lasso['feature'], feat_sorted_lasso['lasso_coef'], color=colors_lasso)
ax1.set_xlabel('Lasso Coefficient (standardized)')
ax1.set_title('Lasso Feature Coefficients\n(0 = feature eliminated by L1)')
ax1.axvline(x=0, color='black', linewidth=0.5)

# RF permutation importance
feat_sorted_rf = feat_imp.sort_values('rf_importance')
ax2.barh(feat_sorted_rf['feature'], feat_sorted_rf['rf_importance'], color='#3498DB')
ax2.set_xlabel('Permutation Importance')
ax2.set_title('Random Forest Permutation Importance')

plt.suptitle('Feature Importance: Two Perspectives', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'nb08_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nFeature Importance Table (sorted by combined rank):')
print(feat_imp[['feature', 'lasso_coef', 'rf_importance', 'univariate_r', 'combined_rank']].to_string(index=False))


Feature Importance Table (sorted by combined rank):
            feature  lasso_coef  rf_importance  univariate_r  combined_rank
     fertility_rate   -3.502233       0.372979     -0.820637            1.0
       abs_latitude    0.837676       0.129702      0.596456            2.0
tertiary_enrollment    0.731696       0.105437      0.613290            3.0
 pm25_air_pollution   -0.632309       0.069384     -0.555517            4.0
 population_density    0.346081       0.013558      0.122502            5.0
       suicide_rate   -0.000000       0.022279      0.311417            6.0
     mean_elevation    0.000000       0.008472     -0.225458            7.0
    forest_area_pct   -0.077660       0.007786      0.088103            8.0
   population_total   -0.000000       0.007880     -0.100395            9.0
      land_area_km2   -0.000000       0.006729      0.045559           10.0


/tmp/ipykernel_1629084/1801605956.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Actual vs Predicted Life Expectancy

In [5]:
fig, ax = plt.subplots(figsize=(12, 10))

bz = residuals[residuals['iso_code'].isin(BLUE_ZONE_ISOS)]
non_bz = residuals[~residuals['iso_code'].isin(BLUE_ZONE_ISOS)]

# Color by classification
for cls, color, marker in [('overperformer', '#27ae60', '^'),
                            ('as_expected', '#7f8c8d', 'o'),
                            ('underperformer', '#c0392b', 'v')]:
    subset = non_bz[non_bz['classification'] == cls]
    ax.scatter(subset['predicted_le'], subset['actual_le'], c=color,
              marker=marker, s=40, alpha=0.6, label=f'{cls} ({len(subset)})')

# Blue Zone countries highlighted
bz_colors = {'JPN': '#3498DB', 'ITA': '#2ECC71', 'GRC': '#9B59B6', 'CRI': '#F39C12', 'USA': '#E74C3C'}
bz_names = {'JPN': 'Japan', 'ITA': 'Italy', 'GRC': 'Greece', 'CRI': 'Costa Rica', 'USA': 'United States'}
for _, row in bz.iterrows():
    ax.scatter(row['predicted_le'], row['actual_le'], c=bz_colors.get(row['iso_code'], 'gold'),
              s=200, marker='*', zorder=10, edgecolors='black', linewidth=0.5)
    ax.annotate(bz_names.get(row['iso_code'], row['iso_code']),
               (row['predicted_le'], row['actual_le']),
               xytext=(8, 5), textcoords='offset points', fontsize=9, fontweight='bold')

# 1:1 line
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', alpha=0.4, label='Perfect prediction (1:1 line)')
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel('Predicted Life Expectancy (years)', fontsize=12)
ax.set_ylabel('Actual Life Expectancy (years)', fontsize=12)
ax.set_title('Actual vs Predicted Life Expectancy (LOOCV)\nPoints above the line = overperformers', fontsize=13)
ax.legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'nb08_actual_vs_predicted.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
r2 = 1 - np.sum((residuals['actual_le'] - residuals['predicted_le'])**2) / np.sum((residuals['actual_le'] - residuals['actual_le'].mean())**2)
rmse = np.sqrt(np.mean((residuals['actual_le'] - residuals['predicted_le'])**2))
print(f'LOOCV R2: {r2:.4f}')
print(f'LOOCV RMSE: {rmse:.2f} years')

LOOCV R2: 0.6832
LOOCV RMSE: 3.56 years


/tmp/ipykernel_1629084/1059610968.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Residual Distribution

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of residuals
ax1.hist(residuals['residual'], bins=20, color='steelblue', edgecolor='white', alpha=0.7)
ax1.axvline(x=0, color='black', linewidth=1)
res_std = residuals['residual'].std()
ax1.axvline(x=res_std, color='green', linewidth=1.5, linestyle='--', label=f'+1 SD ({res_std:.1f} yr)')
ax1.axvline(x=-res_std, color='red', linewidth=1.5, linestyle='--', label=f'-1 SD ({-res_std:.1f} yr)')
ax1.set_xlabel('Residual (Actual - Predicted, years)', fontsize=11)
ax1.set_ylabel('Number of Countries', fontsize=11)
ax1.set_title('Distribution of Prediction Residuals', fontsize=13)
ax1.legend()

# Mark BZ countries
for _, row in bz.iterrows():
    ax1.axvline(x=row['residual'], color=bz_colors.get(row['iso_code'], 'gold'),
               linewidth=1, alpha=0.7)
    ax1.text(row['residual'], ax1.get_ylim()[1] * 0.95,
            bz_names.get(row['iso_code'], ''), fontsize=7, rotation=90,
            va='top', ha='right', color=bz_colors.get(row['iso_code'], 'gold'))

# Z-score distribution
colors_z = ['#27ae60' if z > 1 else '#c0392b' if z < -1 else '#95a5a6'
            for z in residuals['residual_zscore']]
ax2.bar(range(len(residuals)), residuals['residual_zscore'].values, color=colors_z, width=1)
ax2.axhline(y=1, color='green', linewidth=1.5, linestyle='--', label='Overperformer threshold (z=+1)')
ax2.axhline(y=-1, color='red', linewidth=1.5, linestyle='--', label='Underperformer threshold (z=-1)')
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_xlabel('Countries (ranked by residual)', fontsize=11)
ax2.set_ylabel('Residual Z-score', fontsize=11)
ax2.set_title('Residual Z-scores by Country', fontsize=13)
ax2.legend(fontsize=8)

n_over = (residuals['classification'] == 'overperformer').sum()
n_under = (residuals['classification'] == 'underperformer').sum()
n_expected = (residuals['classification'] == 'as_expected').sum()
ax2.text(0.98, 0.02, f'Overperformers: {n_over}\nAs expected: {n_expected}\nUnderperformers: {n_under}',
        transform=ax2.transAxes, fontsize=9, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'nb08_residual_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_1629084/2320924818.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. World Map: Overperformance Choropleth

In [7]:
try:
    import plotly.express as px
    
    fig_map = px.choropleth(
        residuals,
        locations='iso_code',
        color='residual',
        hover_name='country_name',
        hover_data={'actual_le': ':.1f', 'predicted_le': ':.1f', 'residual': ':.2f',
                    'classification': True, 'residual_rank': True},
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        title='Life Expectancy Overperformance Map (Green = lives longer than predicted)',
    )
    fig_map.update_layout(width=1000, height=500, margin=dict(l=0, r=0, t=40, b=0))
    fig_map.write_image(os.path.join(FIGURES_DIR, 'nb08_overperformance_map.png'), scale=2)
    fig_map.show()
    print('Saved: nb08_overperformance_map.png')
except Exception as e:
    print(f'Plotly choropleth not available in this environment: {e}')
    print('Falling back to matplotlib scatter map')
    
    fig, ax = plt.subplots(figsize=(16, 8))
    merged = residuals.merge(features[['iso_code', 'latitude', 'longitude']], on='iso_code', how='left')
    scatter = ax.scatter(merged['longitude'], merged['latitude'],
                        c=merged['residual'], cmap='RdYlGn', s=60,
                        edgecolors='black', linewidth=0.3, vmin=-10, vmax=10)
    plt.colorbar(scatter, ax=ax, label='Residual (years)', shrink=0.7)
    
    # Label BZ countries
    bz_merged = merged[merged['iso_code'].isin(BLUE_ZONE_ISOS)]
    for _, row in bz_merged.iterrows():
        ax.annotate(bz_names.get(row['iso_code'], ''),
                   (row['longitude'], row['latitude']),
                   xytext=(5, 5), textcoords='offset points', fontsize=8, fontweight='bold')
    
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Life Expectancy Overperformance Map\n(Green = lives longer than predicted, Red = shorter)', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'nb08_overperformance_map.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: nb08_overperformance_map.png (scatter fallback)')

Plotly choropleth not available in this environment: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

Falling back to matplotlib scatter map
Saved: nb08_overperformance_map.png (scatter fallback)


/tmp/ipykernel_1629084/3788716500.py:42: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



## 7. Top 15 Overperformers ("Hidden Blue Zone" Candidates)

In [8]:
print('TOP 15 OVERPERFORMERS')
print('Countries whose actual life expectancy significantly exceeds prediction')
print('=' * 90)
display_cols = ['residual_rank', 'country_name', 'is_blue_zone', 'actual_le', 'predicted_le', 'residual', 'residual_zscore']
print(hidden_bz[display_cols].to_string(index=False))

n_bz_in_top = hidden_bz['is_blue_zone'].sum()
print(f'\nKnown Blue Zone countries in top 15: {n_bz_in_top}')
print(f'Non-BZ overperformers ("hidden Blue Zone" candidates): {len(hidden_bz) - n_bz_in_top}')

TOP 15 OVERPERFORMERS
Countries whose actual life expectancy significantly exceeds prediction
 residual_rank country_name  is_blue_zone  actual_le  predicted_le  residual  residual_zscore
             1       Israel             0  83.195122         73.39      9.81            2.755
             2        Japan             1  84.041220         78.20      5.84            1.646
             3 Saudi Arabia             0  78.732000         72.97      5.76            1.625
             4   Luxembourg             0  83.358537         78.19      5.17            1.459
             5    Singapore             0  82.895122         77.81      5.09            1.437
             6  South Korea             0  83.429268         78.44      4.99            1.411
             7       Jordan             0  77.814000         72.98      4.83            1.366
             8   Costa Rica             1  80.799000         76.12      4.68            1.323
             9         Peru             0  77.740000        

## 8. Bottom 15 Underperformers

In [9]:
print('BOTTOM 15 UNDERPERFORMERS')
print('Countries whose actual life expectancy falls significantly below prediction')
print('=' * 90)
print(underperf[display_cols].to_string(index=False))

BOTTOM 15 UNDERPERFORMERS
Countries whose actual life expectancy falls significantly below prediction
 residual_rank     country_name  is_blue_zone  actual_le  predicted_le  residual  residual_zscore
            79       Mozambique             0  63.611000         67.28     -3.67           -1.008
            80        Lithuania             0  76.987805         80.69     -3.70           -1.017
            81            Libya             0  69.339000         73.25     -3.91           -1.077
            82      Philippines             0  69.833000         73.91     -4.07           -1.121
            83            Kenya             0  63.646000         67.77     -4.13           -1.137
            84      Afghanistan             0  66.035000         70.39     -4.36           -1.202
            85           Latvia             0  75.678049         80.10     -4.42           -1.219
            86          Myanmar             0  66.889000         71.35     -4.47           -1.231
            87  

## 9. Blue Zone Validation

Do the known Blue Zone countries appear as overperformers? If the model is
capturing something real, countries containing Blue Zones should tend to
outperform their predictions.

In [10]:
bz_results = residuals[residuals['iso_code'].isin(BLUE_ZONE_ISOS)].copy()

print('BLUE ZONE VALIDATION')
print('=' * 80)
print(bz_results[display_cols + ['classification']].to_string(index=False))

print(f'\nBZ mean residual: {bz_results["residual"].mean():+.2f} years')
print(f'BZ mean z-score: {bz_results["residual_zscore"].mean():+.3f}')
print(f'BZ median rank: {bz_results["residual_rank"].median():.0f}/93')

# Excluding USA
bz_no_usa = bz_results[bz_results['iso_code'] != 'USA']
print(f'\nExcluding USA:')
print(f'  Mean residual: {bz_no_usa["residual"].mean():+.2f} years')
print(f'  All in top half: {(bz_no_usa["residual_rank"] <= 47).all()}')
print(f'  Classified as overperformers: {(bz_no_usa["classification"] == "overperformer").sum()}/{len(bz_no_usa)}')

print(f'\nUSA specifically:')
usa = bz_results[bz_results['iso_code'] == 'USA'].iloc[0]
print(f'  Rank: {usa["residual_rank"]}/93')
print(f'  Residual: {usa["residual"]:+.2f} years (lives {abs(usa["residual"]):.1f} years less than predicted)')
print(f'  Classification: {usa["classification"]}')

BLUE ZONE VALIDATION
 residual_rank  country_name  is_blue_zone  actual_le  predicted_le  residual  residual_zscore classification
             2         Japan             1  84.041220         78.20      5.84            1.646  overperformer
             8    Costa Rica             1  80.799000         76.12      4.68            1.323  overperformer
            10         Italy             1  83.700000         79.42      4.28            1.210  overperformer
            27        Greece             1  81.536585         79.64      1.89            0.545    as_expected
            69 United States             1  78.385366         80.47     -2.09           -0.566    as_expected

BZ mean residual: +2.92 years
BZ mean z-score: +0.832
BZ median rank: 10/93

Excluding USA:
  Mean residual: +4.17 years
  All in top half: True
  Classified as overperformers: 3/4

USA specifically:
  Rank: 69/93
  Residual: -2.09 years (lives 2.1 years less than predicted)
  Classification: as_expected


## 10. Residual vs GDP (Income Patterns in Overperformance)

In [11]:
fig, ax = plt.subplots(figsize=(14, 8))

merged_gdp = residuals.merge(features[['iso_code', 'gdp_per_capita']], on='iso_code', how='left')

non_bz_gdp = merged_gdp[~merged_gdp['iso_code'].isin(BLUE_ZONE_ISOS)]
bz_gdp = merged_gdp[merged_gdp['iso_code'].isin(BLUE_ZONE_ISOS)]

colors_cls = {'overperformer': '#27ae60', 'as_expected': '#95a5a6', 'underperformer': '#c0392b'}
for cls in ['underperformer', 'as_expected', 'overperformer']:
    subset = non_bz_gdp[non_bz_gdp['classification'] == cls]
    ax.scatter(subset['gdp_per_capita'], subset['residual'], c=colors_cls[cls],
              alpha=0.6, s=40, label=f'{cls} ({len(subset)})')

for _, row in bz_gdp.iterrows():
    ax.scatter(row['gdp_per_capita'], row['residual'],
              c=bz_colors.get(row['iso_code'], 'gold'),
              s=200, marker='*', zorder=10, edgecolors='black', linewidth=0.5)
    ax.annotate(bz_names.get(row['iso_code'], row['iso_code']),
               (row['gdp_per_capita'], row['residual']),
               xytext=(8, 5), textcoords='offset points', fontsize=9, fontweight='bold')

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('GDP per Capita (USD)', fontsize=12)
ax.set_ylabel('Residual (Actual - Predicted LE, years)', fontsize=12)
ax.set_title('Overperformance vs GDP per Capita\n(Above 0 = lives longer than predicted)', fontsize=13)
ax.set_xscale('log')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'nb08_residual_vs_gdp.png'), dpi=150, bbox_inches='tight')
plt.show()

r, p = stats.pearsonr(merged_gdp['gdp_per_capita'].dropna(), 
                       merged_gdp.loc[merged_gdp['gdp_per_capita'].notna(), 'residual'])
print(f'Correlation between GDP and residual: r={r:.3f}, p={p:.4f}')
print('A low correlation means the model already accounts for GDP-related effects.')

Correlation between GDP and residual: r=0.426, p=0.0000
A low correlation means the model already accounts for GDP-related effects.


/tmp/ipykernel_1629084/2135210431.py:31: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



## 11. Residual vs Latitude (Geographic Patterns)

In [12]:
fig, ax = plt.subplots(figsize=(14, 8))

merged_lat = residuals.merge(features[['iso_code', 'latitude']], on='iso_code', how='left')

non_bz_lat = merged_lat[~merged_lat['iso_code'].isin(BLUE_ZONE_ISOS)]
bz_lat = merged_lat[merged_lat['iso_code'].isin(BLUE_ZONE_ISOS)]

for cls in ['underperformer', 'as_expected', 'overperformer']:
    subset = non_bz_lat[non_bz_lat['classification'] == cls]
    ax.scatter(subset['latitude'], subset['residual'], c=colors_cls[cls],
              alpha=0.6, s=40, label=f'{cls} ({len(subset)})')

for _, row in bz_lat.iterrows():
    ax.scatter(row['latitude'], row['residual'],
              c=bz_colors.get(row['iso_code'], 'gold'),
              s=200, marker='*', zorder=10, edgecolors='black', linewidth=0.5)
    ax.annotate(bz_names.get(row['iso_code'], row['iso_code']),
               (row['latitude'], row['residual']),
               xytext=(8, 5), textcoords='offset points', fontsize=9, fontweight='bold')

ax.axhline(y=0, color='black', linewidth=0.5)
ax.set_xlabel('Latitude (degrees)', fontsize=12)
ax.set_ylabel('Residual (Actual - Predicted LE, years)', fontsize=12)
ax.set_title('Overperformance vs Latitude\n(Geographic patterns in unexplained longevity)', fontsize=13)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'nb08_residual_vs_latitude.png'), dpi=150, bbox_inches='tight')
plt.show()

/tmp/ipykernel_1629084/2027424366.py:29: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



## 12. Summary and Caveats

In [13]:
best = model_comp.iloc[0]

print('ML PREDICTION AND OVERPERFORMANCE ANALYSIS -- SUMMARY')
print('=' * 60)
print()
print('APPROACH:')
print(f'  Features: 10 (selected from 26 via coverage/correlation/VIF pipeline)')
print(f'  Models: 5 (OLS, Ridge, Lasso, ElasticNet, Random Forest)')
print(f'  Evaluation: Leave-One-Out Cross-Validation (n=93)')
print(f'  Best model: {best["model"]} (LOOCV R2={best["loocv_r2"]:.4f}, RMSE={best["loocv_rmse"]:.2f} yr)')
print()
print('KEY FINDINGS:')
print(f'  1. Fertility rate is the strongest predictor of life expectancy')
print(f'     (Lasso coef: {feat_imp.iloc[0]["lasso_coef"]:.2f}, RF importance: {feat_imp.iloc[0]["rf_importance"]:.3f})')
print(f'  2. Blue Zone countries (excl. USA) are all in the top half of residual rankings')
print(f'     Japan (rank 2), Costa Rica (8), Italy (10) are confirmed overperformers')
print(f'  3. USA ranks {int(usa["residual_rank"])}/93 -- underperforms its predicted LE by {abs(usa["residual"]):.1f} years')
print(f'  4. Top "hidden Blue Zone" candidates: Israel, Saudi Arabia, Luxembourg, Singapore, South Korea')
print(f'  5. Worst underperformers: Nigeria, Russia, Fiji, South Africa')
print()
print('CAVEATS:')
print('  - Small sample (n=93): results should be treated as exploratory')
print('  - Cross-sectional only: no causal claims possible')
print('  - Some important features (obesity, NCD mortality) unavailable from API')
print('  - Country-level data masks sub-national variation (the Loma Linda problem)')
print('  - Residuals reflect unmeasured factors AND measurement error')
print('  - "Hidden Blue Zone" is a label for overperformance, not a clinical designation')

ML PREDICTION AND OVERPERFORMANCE ANALYSIS -- SUMMARY

APPROACH:
  Features: 10 (selected from 26 via coverage/correlation/VIF pipeline)
  Models: 5 (OLS, Ridge, Lasso, ElasticNet, Random Forest)
  Evaluation: Leave-One-Out Cross-Validation (n=93)
  Best model: Random Forest (LOOCV R2=0.6832, RMSE=3.56 yr)

KEY FINDINGS:
  1. Fertility rate is the strongest predictor of life expectancy
     (Lasso coef: -3.50, RF importance: 0.373)
  2. Blue Zone countries (excl. USA) are all in the top half of residual rankings
     Japan (rank 2), Costa Rica (8), Italy (10) are confirmed overperformers
  3. USA ranks 69/93 -- underperforms its predicted LE by 2.1 years
  4. Top "hidden Blue Zone" candidates: Israel, Saudi Arabia, Luxembourg, Singapore, South Korea
  5. Worst underperformers: Nigeria, Russia, Fiji, South Africa

CAVEATS:
  - Small sample (n=93): results should be treated as exploratory
  - Cross-sectional only: no causal claims possible
  - Some important features (obesity, NCD mortal